# 03 — Grad-CAM

Runs multi-layer Grad-CAM on examples selected from each outcome category
(TP, FP, FN, TN) and overlays the activation map on the natural-color tile.

Optionally adds an occlusion sensitivity map alongside each Grad-CAM for
a gradient-free cross-check.

Requires `02_evaluation.ipynb` to have been run (needs `val.csv`).

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CKPT_PATH   = "/path/to/model_states/best.pt"
PROJECT_DIR = "/path/to/ct_classifier"
FIGURES_DIR = "figures"

N_PER_CATEGORY = 3      # examples to show per TP/FP/FN/TN
CAM_ALPHA      = 0.45   # heatmap opacity
RUN_OCCLUSION  = True   # set False to skip the slower occlusion maps
SEED           = 0

In [ ]:
import os, sys, random
import yaml
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from ct_classifier.dataset import BleachDataset
from ct_classifier.model import CustomResNet
from utils.evaluate import get_model_preds
from utils.viz import to_display_rgb, percentile_stretch, overlay_cam
from utils.gradcam import GradCAM, make_occlusion_map

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
# ── Load config + model ───────────────────────────────────────────────────────
cfg_path = os.path.join(os.path.dirname(CKPT_PATH), "config.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CustomResNet(cfg["num_classes"], cfg["layers"])
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state["model"])
model.to(device).eval()

# Target layers for multi-layer Grad-CAM (layer3 + layer4 averaged)
target_layers = [
    model.feature_extractor.layer3,
    model.feature_extractor.layer4,
]
cam_extractor = GradCAM(model, target_layers)

In [ ]:
# ── Run inference to get TP/FP/FN/TN labels ──────────────────────────────────
dl_val = DataLoader(
    BleachDataset(cfg, split="val", eval=True),
    batch_size=cfg.get("batch_size", 32),
    shuffle=False,
    num_workers=cfg.get("num_workers", 0),
)
val_df = get_model_preds(model, dl_val, device=device)

val_df["pred"] = (val_df["softmax_bleach_scores"] > 0.5).astype(int)
val_df["outcome"] = val_df.apply(
    lambda r: {
        (1, 1): "TP", (0, 0): "TN",
        (0, 1): "FP", (1, 0): "FN",
    }[(r["pred"], r["labels"])], axis=1
)
val_df["outcome"].value_counts()

In [ ]:
# ── Grad-CAM grid ─────────────────────────────────────────────────────────────
ds_val = dl_val.dataset

def get_tile(image_id):
    """Fetch the raw [18,H,W] tensor for a given image_id."""
    row_idx = ds_val.meta.query("image_id == @image_id").index[0]
    # find the couple containing this image_id as the first member
    for i, (id1, id2) in enumerate(ds_val.image_couples):
        if id1 == image_id:
            tile, _, _ = ds_val[i]
            return tile
    return None

categories = ["TP", "FP", "FN", "TN"]
cols = 3 if not RUN_OCCLUSION else 4   # RGB | CAM | (OCC) | score text

for cat in categories:
    subset = val_df[val_df["outcome"] == cat]
    if subset.empty:
        print(f"No {cat} examples in val set.")
        continue

    samples = subset.sample(min(N_PER_CATEGORY, len(subset)), random_state=SEED)
    n = len(samples)
    ncols = 3 + RUN_OCCLUSION
    fig, axes = plt.subplots(n, ncols, figsize=(ncols * 3.5, n * 3.5))
    if n == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(f"{cat} examples", fontsize=13, y=1.01)

    for row, (_, r) in enumerate(samples.iterrows()):
        tile = get_tile(int(r["image_id1"]))
        if tile is None:
            continue

        rgb = percentile_stretch(to_display_rgb(tile[:8]))
        cam = cam_extractor(tile.to(device), class_idx="pred")

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f"RGB  gt={r['str_labels']}")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(rgb)
        axes[row, 1].imshow(plt.cm.jet(cam)[..., :3], alpha=CAM_ALPHA)
        axes[row, 1].set_title(f"Grad-CAM  p={r['softmax_bleach_scores']:.2f}")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(overlay_cam(rgb, cam, alpha=CAM_ALPHA))
        axes[row, 2].set_title("Blended")
        axes[row, 2].axis("off")

        if RUN_OCCLUSION:
            occ = make_occlusion_map(model, tile, device=device)
            axes[row, 3].imshow(overlay_cam(rgb, occ, alpha=CAM_ALPHA))
            axes[row, 3].set_title("Occlusion")
            axes[row, 3].axis("off")

    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"gradcam_{cat}.png"), dpi=150, bbox_inches="tight")
    plt.show()

cam_extractor.remove_hooks()